# 0405: Gaussian Model (가우시안 모델)

## 개요 (Overview)

본 문서에서는 `scene/gaussian_model.py`를 분석하여 다음을 이해합니다:

This document analyzes `scene/gaussian_model.py` to understand:
- 3D 가우시안의 수학적 표현 (Mathematical formulation of 3D Gaussians)
- 매개변수 저장 및 활성화 함수 (Parameter storage and activation functions)
- Spherical Harmonics로 색상 표현 (Color representation with Spherical Harmonics)
- COLMAP으로부터 초기화 (Initialization from COLMAP)
- Densification 연산 구현 (Implementation of densification operations)

---

## 1. 3D 가우시안 수학적 표현 (Mathematical Formulation)

### 1.1 기본 3D 가우시안 함수 (Basic 3D Gaussian Function)

3D 가우시안은 3차원 공간의 점 $\mathbf{x}$에서 다음과 같이 정의됩니다:

$$
G(\mathbf{x}) = e^{-\frac{1}{2}(\mathbf{x}-\boldsymbol{\mu})^T \Sigma^{-1} (\mathbf{x}-\boldsymbol{\mu})}
$$

여기서:
- $\boldsymbol{\mu} \in \mathbb{R}^3$: 평균 (중심 위치, mean/center position)
- $\Sigma \in \mathbb{R}^{3 \times 3}$: 공분산 행렬 (covariance matrix)

**물리적 의미**:
- $\boldsymbol{\mu}$: 가우시안의 중심 위치 (3DGS에서는 `_xyz` 파라미터)
- $\Sigma$: 가우시안의 "모양"을 결정 (크기, 방향, 비등방성)

---

### 1.2 공분산 행렬 분해 (Covariance Matrix Decomposition)

공분산 행렬 $\Sigma$를 직접 학습하면 문제가 있습니다:
1. **양정부호(positive semi-definite) 제약**: $\Sigma$는 항상 PSD 행렬이어야 함
2. **9개 파라미터**: 대칭 행렬이지만 6개 자유도만 필요

**해결책**: 공분산을 **스케일 + 회전**으로 분해

$$
\Sigma = R S S^T R^T
$$

여기서:
- $S = \text{diag}(s_x, s_y, s_z) \in \mathbb{R}^{3 \times 3}$: 스케일 행렬 (scale matrix)
- $R \in SO(3)$: 회전 행렬 (rotation matrix)

**파라미터 최소화**:
- $\mathbf{s} = [s_x, s_y, s_z]^T$: 3D 스케일 (3 parameters)
- $\mathbf{q} = [q_w, q_x, q_y, q_z]^T$: 단위 쿼터니언 (4 parameters for rotation)
- **총 7개 파라미터로 공분산 행렬 표현**

---

### 1.3 코드 구현 (Code Implementation)

`gaussian_model.py`의 `setup_functions()`:

In [ ]:
def build_covariance_from_scaling_rotation(scaling, scaling_modifier, rotation):
    L = build_scaling_rotation(scaling_modifier * scaling, rotation)
    actual_covariance = L @ L.transpose(1, 2)
    symm = strip_symmetric(actual_covariance)
    return symm

self.covariance_activation = build_covariance_from_scaling_rotation



**단계별 분석**:
1. `build_scaling_rotation(s, R)` → $L = R \cdot S$ (행렬 곱)
2. `L @ L.transpose(1, 2)` → $\Sigma = L L^T = R S S^T R^T$
3. `strip_symmetric(Σ)` → 대칭 행렬을 6개 원소로 압축 (메모리 최적화)

`utils/general_utils.py`의 `build_scaling_rotation`:

In [ ]:
def build_scaling_rotation(s, r):
    L = torch.zeros((s.shape[0], 3, 3), dtype=torch.float, device="cuda")
    R = build_rotation(r)  # 쿼터니언 → 회전 행렬

    L[:,0,0] = s[:,0]
    L[:,1,1] = s[:,1]
    L[:,2,2] = s[:,2]

    L = R @ L  # (N, 3, 3) @ (N, 3, 3)
    return L



---

## 2. 가우시안 파라미터 저장 (Gaussian Parameter Storage)

### 2.1 내부 파라미터 (`_` prefix)

`__init__` 메서드에서 초기화되는 파라미터들:

In [ ]:
def __init__(self, sh_degree, optimizer_type="default"):
    self.active_sh_degree = 0
    self.optimizer_type = optimizer_type
    self.max_sh_degree = sh_degree  
    self._xyz = torch.empty(0)           # 위치 (position)
    self._features_dc = torch.empty(0)   # DC SH 계수 (DC spherical harmonics)
    self._features_rest = torch.empty(0) # 고차 SH 계수 (higher-order SH)
    self._scaling = torch.empty(0)       # 스케일 (scale, log space)
    self._rotation = torch.empty(0)      # 회전 (rotation, quaternion)
    self._opacity = torch.empty(0)       # 불투명도 (opacity, logit space)
    self.max_radii2D = torch.empty(0)    # 2D 화면 반경 (screen-space radius)
    self.xyz_gradient_accum = torch.empty(0)  # Gradient 누적
    self.denom = torch.empty(0)          # Gradient 평균 계산용
    self.optimizer = None
    self.percent_dense = 0
    self.spatial_lr_scale = 0
    self.setup_functions()



**중요**: 파라미터는 **변환된 공간(transformed space)**에 저장됩니다:
- `_scaling`: $\log(\mathbf{s})$ 형태 (음수 방지)
- `_opacity`: $\text{logit}(\alpha)$ 형태 (0-1 범위 보장)
- `_rotation`: 정규화되지 않은 쿼터니언 (최적화 중 자동 정규화)

---

### 2.2 활성화 함수 (Activation Functions)

`setup_functions()`:

In [ ]:
self.scaling_activation = torch.exp
self.scaling_inverse_activation = torch.log

self.opacity_activation = torch.sigmoid
self.inverse_opacity_activation = inverse_sigmoid

self.rotation_activation = torch.nn.functional.normalize



**변환 과정**:
1. **스케일**: $\mathbf{s} = \exp(\mathbf{\_scaling})$ → 항상 양수
2. **불투명도**: $\alpha = \sigma(\mathbf{\_opacity})$ → 범위 [0, 1]
3. **회전**: $\mathbf{q} = \frac{\mathbf{\_rotation}}{||\mathbf{\_rotation}||}$ → 단위 쿼터니언

---

### 2.3 Property Accessors

외부에서 파라미터 접근 시 자동으로 활성화 함수 적용:

In [ ]:
@property
def get_scaling(self):
    return self.scaling_activation(self._scaling)  # exp(log(s)) = s

@property
def get_rotation(self):
    return self.rotation_activation(self._rotation)  # normalize(q)

@property
def get_opacity(self):
    return self.opacity_activation(self._opacity)  # sigmoid(logit(α))

@property
def get_xyz(self):
    return self._xyz  # 위치는 직접 저장



**사용 예시**:

In [ ]:
# train.py에서
xyz = gaussians.get_xyz
rotation = gaussians.get_rotation  # 자동으로 정규화됨
scaling = gaussians.get_scaling    # 자동으로 exp 적용



---

## 3. Spherical Harmonics 색상 표현 (Color Representation)

### 3.1 Spherical Harmonics란? (What are Spherical Harmonics?)

**문제**: 가우시안의 색상이 **시점(viewing direction)**에 따라 달라져야 합니다 (view-dependent color).

**해결책**: Spherical Harmonics (SH) 기저 함수로 방향에 따른 색상 함수를 표현:

$$
C(\mathbf{d}) = \sum_{l=0}^{l_{\max}} \sum_{m=-l}^{l} k_l^m \cdot Y_l^m(\mathbf{d})
$$

여기서:
- $\mathbf{d}$: 시선 방향 (viewing direction, 단위 벡터)
- $Y_l^m(\mathbf{d})$: SH 기저 함수 (degree $l$, order $m$)
- $k_l^m$: SH 계수 (학습 가능한 파라미터)
- $l_{\max}$: 최대 degree (3DGS에서는 3)

---

### 3.2 3DGS의 SH 계수 저장 (SH Coefficient Storage in 3DGS)

**파라미터 개수** (RGB 각 채널당):
- Degree 0: 1개 (DC component)
- Degree 1: 3개
- Degree 2: 5개
- Degree 3: 7개
- **총 16개 계수** (degree 3까지)

**저장 방식**:

In [ ]:
# __init__
self._features_dc = torch.empty(0)    # Shape: (N, 1, 3) - DC만
self._features_rest = torch.empty(0)  # Shape: (N, 15, 3) - 나머지

# create_from_pcd에서
features = torch.zeros((N, 3, 16)).float().cuda()  # (N, RGB, 16 coeffs)
features[:, :3, 0] = fused_color  # DC 초기화
features[:, 3:, 1:] = 0.0         # 고차 계수는 0으로

self._features_dc = nn.Parameter(features[:,:,0:1].transpose(1, 2))  # (N,1,3)
self._features_rest = nn.Parameter(features[:,:,1:].transpose(1, 2)) # (N,15,3)



**왜 분리하나?** (Why separate DC and rest?)
- DC는 평균 색상 (constant term) → 학습률 높게 (`feature_lr`)
- 고차 계수는 방향 의존성 → 학습률 낮게 (`feature_lr / 20.0`)

---

### 3.3 Progressive SH Degree Training

SH degree는 점진적으로 증가합니다:

In [ ]:
# train.py
if iteration % 1000 == 0:
    gaussians.oneupSHdegree()

# gaussian_model.py
def oneupSHdegree(self):
    if self.active_sh_degree < self.max_sh_degree:
        self.active_sh_degree += 1



**학습 스케줄**:
- 0-1000 iter: degree 0 (DC only)
- 1000-2000 iter: degree 1 (DC + linear)
- 2000-3000 iter: degree 2 (DC + linear + quadratic)
- 3000+ iter: degree 3 (full 16 coefficients)

**이유**:
- 초기에는 평균 색상만 학습 (안정적 수렴)
- 점차 복잡한 view-dependent 효과 학습

---

### 3.4 전체 Feature Vector 접근

In [ ]:
@property
def get_features(self):
    features_dc = self._features_dc
    features_rest = self._features_rest
    return torch.cat((features_dc, features_rest), dim=1)  # (N, 16, 3)



---

## 4. COLMAP으로부터 초기화 (Initialization from COLMAP)

### 4.1 `create_from_pcd` 메서드

`points3D.ply`로부터 가우시안 초기화:

In [ ]:
def create_from_pcd(self, pcd : BasicPointCloud, cam_infos : int, spatial_lr_scale : float):
    self.spatial_lr_scale = spatial_lr_scale
    fused_point_cloud = torch.tensor(np.asarray(pcd.points)).float().cuda()
    fused_color = RGB2SH(torch.tensor(np.asarray(pcd.colors)).float().cuda())



**입력**:
- `pcd.points`: COLMAP sparse reconstruction 점들 (N × 3)
- `pcd.colors`: 각 점의 RGB 색상 (N × 3, 범위 [0, 1])
- `spatial_lr_scale`: Scene extent 기반 학습률 조정

---

### 4.2 초기 스케일 결정 (Initial Scale Determination)

**핵심 아이디어**: 각 점의 스케일을 **가장 가까운 이웃까지의 거리**로 설정

In [ ]:
from simple_knn._C import distCUDA2

dist2 = torch.clamp_min(distCUDA2(torch.from_numpy(np.asarray(pcd.points)).float().cuda()), 0.0000001)
scales = torch.log(torch.sqrt(dist2))[...,None].repeat(1, 3)



**단계별 분석**:
1. `distCUDA2(points)` → 각 점에서 가장 가까운 3개 이웃까지의 평균 제곱거리 $d^2$
2. `torch.sqrt(dist2)` → $d$ (실제 거리)
3. `torch.log(d)` → log space 변환 (저장용)
4. `.repeat(1, 3)` → $(s_x, s_y, s_z)$ 모두 같은 값으로 초기화 (등방성 가우시안)

**왜 이웃 거리를 사용하나?**
- 점들 사이 빈 공간을 채우기 위해
- COLMAP sparse points는 서로 너무 가까우면 중복, 너무 멀면 빈 공간 발생
- 이웃 거리 ≈ 적절한 초기 반경

---

### 4.3 초기 회전 및 불투명도 (Initial Rotation and Opacity)

In [ ]:
rots = torch.zeros((N, 4), device="cuda")
rots[:, 0] = 1  # 쿼터니언 [1, 0, 0, 0] = identity rotation

opacities = self.inverse_opacity_activation(0.1 * torch.ones((N, 1)))
# inverse_sigmoid(0.1) ≈ -2.197



**초기값 설정**:
- **회전**: Identity (회전 없음) → 학습 중 최적화
- **불투명도**: 0.1 (10%) → 너무 높으면 초기에 불투명해서 학습 어려움

---

### 4.4 Spherical Harmonics 초기화

In [ ]:
features = torch.zeros((fused_color.shape[0], 3, (self.max_sh_degree + 1) ** 2)).float().cuda()
features[:, :3, 0 ] = fused_color  # DC coefficient = COLMAP 색상
features[:, 3:, 1:] = 0.0          # 나머지는 0



**DC 계수 초기화**:
- COLMAP RGB → SH DC coefficient (using `RGB2SH` utility)
- 이후 트레이닝 중 고차 계수가 학습됨

---

### 4.5 nn.Parameter 등록

In [ ]:
self._xyz = nn.Parameter(fused_point_cloud.requires_grad_(True))
self._features_dc = nn.Parameter(features[:,:,0:1].transpose(1, 2).contiguous().requires_grad_(True))
self._features_rest = nn.Parameter(features[:,:,1:].transpose(1, 2).contiguous().requires_grad_(True))
self._scaling = nn.Parameter(scales.requires_grad_(True))
self._rotation = nn.Parameter(rots.requires_grad_(True))
self._opacity = nn.Parameter(opacities.requires_grad_(True))
self.max_radii2D = torch.zeros((self.get_xyz.shape[0]), device="cuda")



PyTorch의 `nn.Parameter`로 등록하면:
- 자동으로 optimizer에 추가됨
- `.requires_grad_(True)`로 gradient 계산 활성화

---

## 5. Training Setup (트레이닝 설정)

### 5.1 학습률 설정 (Learning Rate Configuration)

`training_setup` 메서드:

In [ ]:
def training_setup(self, training_args):
    self.percent_dense = training_args.percent_dense
    self.xyz_gradient_accum = torch.zeros((self.get_xyz.shape[0], 1), device="cuda")
    self.denom = torch.zeros((self.get_xyz.shape[0], 1), device="cuda")

    l = [
        {'params': [self._xyz], 'lr': training_args.position_lr_init * self.spatial_lr_scale, "name": "xyz"},
        {'params': [self._features_dc], 'lr': training_args.feature_lr, "name": "f_dc"},
        {'params': [self._features_rest], 'lr': training_args.feature_lr / 20.0, "name": "f_rest"},
        {'params': [self._opacity], 'lr': training_args.opacity_lr, "name": "opacity"},
        {'params': [self._scaling], 'lr': training_args.scaling_lr, "name": "scaling"},
        {'params': [self._rotation], 'lr': training_args.rotation_lr, "name": "rotation"}
    ]



**하이퍼파라미터** (기본값):
| Parameter | Learning Rate | 이유 (Reason) |
|-----------|---------------|---------------|
| `xyz` | `0.00016 × spatial_lr_scale` | Scene 크기에 비례 조정 |
| `f_dc` | `0.0025` | 평균 색상은 빠르게 학습 |
| `f_rest` | `0.000125` (1/20) | View-dependent 효과는 천천히 |
| `opacity` | `0.05` | 불투명도 변화는 민감함 |
| `scaling` | `0.005` | 스케일 조정 |
| `rotation` | `0.001` | 회전은 섬세하게 조정 |

---

### 5.2 Optimizer 선택

In [ ]:
if self.optimizer_type == "default":
    self.optimizer = torch.optim.Adam(l, lr=0.0, eps=1e-15)
elif self.optimizer_type == "sparse_adam":
    try:
        self.optimizer = SparseGaussianAdam(l, lr=0.0, eps=1e-15)
    except:
        self.optimizer = torch.optim.Adam(l, lr=0.0, eps=1e-15)



**Sparse Adam**:
- 커스텀 optimizer (특수 빌드 필요)
- Densification 후 optimizer state 효율적 관리
- 사용 불가능하면 자동으로 standard Adam 사용

**왜 `lr=0.0`?**
- 각 parameter group에 이미 학습률 설정됨
- Global learning rate는 무시됨
- `update_learning_rate()`에서 동적으로 조정

---

## 6. Densification Operations (밀집화 연산)

clone/split/prune을 구현합니다.

### 6.1 Gradient 누적 (Gradient Accumulation)

In [ ]:
def add_densification_stats(self, viewspace_point_tensor, update_filter):
    self.xyz_gradient_accum[update_filter] += torch.norm(viewspace_point_tensor.grad[update_filter,:2], dim=-1, keepdim=True)
    self.denom[update_filter] += 1



**역할**:
- `viewspace_point_tensor.grad[:, :2]` → 2D screen space gradient $(g_x, g_y)$
- $||\nabla_{xy}|| = \sqrt{g_x^2 + g_y^2}$ → 2D gradient 크기
- 100 iterations마다 누적하여 평균 gradient 계산

---

### 6.2 Clone 연산 (Cloning Small Gaussians)

**조건**: 작은 가우시안 + 높은 gradient

In [ ]:
def densify_and_clone(self, grads, grad_threshold, scene_extent):
    # Extract points that satisfy the gradient condition
    selected_pts_mask = torch.where(torch.norm(grads, dim=-1) >= grad_threshold, True, False)
    selected_pts_mask = torch.logical_and(selected_pts_mask,
                                          torch.max(self.get_scaling, dim=1).values <= self.percent_dense*scene_extent)
    
    new_xyz = self._xyz[selected_pts_mask]
    new_features_dc = self._features_dc[selected_pts_mask]
    new_features_rest = self._features_rest[selected_pts_mask]
    new_opacities = self._opacity[selected_pts_mask]
    new_scaling = self._scaling[selected_pts_mask]
    new_rotation = self._rotation[selected_pts_mask]

    new_tmp_radii = self.tmp_radii[selected_pts_mask]

    self.densification_postfix(new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling, new_rotation, new_tmp_radii)



**핵심 아이디어**:
- 작은 가우시안은 **복사(duplicate)** → 같은 위치에 2개 가우시안
- 모든 속성이 동일하게 복사됨
- 이후 학습으로 두 가우시안이 분리되며 underreconstructed 영역 채움

---

### 6.3 Split 연산 (Splitting Large Gaussians)

**조건**: 큰 가우시안 + 높은 gradient

In [ ]:
def densify_and_split(self, grads, grad_threshold, scene_extent, N=2):
    n_init_points = self.get_xyz.shape[0]
    padded_grad = torch.zeros((n_init_points), device="cuda")
    padded_grad[:grads.shape[0]] = grads.squeeze()
    selected_pts_mask = torch.where(padded_grad >= grad_threshold, True, False)
    selected_pts_mask = torch.logical_and(selected_pts_mask,
                                          torch.max(self.get_scaling, dim=1).values > self.percent_dense*scene_extent)

    stds = self.get_scaling[selected_pts_mask].repeat(N,1)
    means = torch.zeros((stds.size(0), 3), device="cuda")
    samples = torch.normal(mean=means, std=stds)
    rots = build_rotation(self._rotation[selected_pts_mask]).repeat(N,1,1)
    new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1) + self.get_xyz[selected_pts_mask].repeat(N, 1)
    new_scaling = self.scaling_inverse_activation(self.get_scaling[selected_pts_mask].repeat(N,1) / (0.8*N))



**단계별 분석**:

1. **가우시안 샘플링**:

In [ ]:
   samples = torch.normal(mean=0, std=stds)
   # samples ~ N(0, Σ) in local space
   



2. **로컬 → 월드 좌표 변환**:

In [ ]:
   rots = build_rotation(self._rotation[selected_pts_mask])  # 쿼터니언 → 회전 행렬
   new_xyz = torch.bmm(rots, samples.unsqueeze(-1)).squeeze(-1) + original_xyz
   # new_xyz = R × sample + μ
   



3. **스케일 축소**:

In [ ]:
   new_scaling = log(original_scale / (0.8 * N))
   # N=2일 때: new_scale = original_scale / 1.6
   



4. **부모 제거**:

In [ ]:
   prune_filter = torch.cat((selected_pts_mask, torch.zeros(N * selected_pts_mask.sum(), device="cuda", dtype=bool)))
   self.prune_points(prune_filter)
   



---

### 6.4 Prune 연산 (Pruning Unnecessary Gaussians)

**조건**:
1. 불투명도 < 0.005 (거의 투명)
2. 화면 크기 > 20px (너무 큰 2D projection)
3. 월드 크기 > 0.1 × scene_extent (너무 큰 3D Gaussian)

In [ ]:
def densify_and_prune(self, max_grad, min_opacity, extent, max_screen_size, radii):
    grads = self.xyz_gradient_accum / self.denom
    grads[grads.isnan()] = 0.0

    self.tmp_radii = radii
    self.densify_and_clone(grads, max_grad, extent)
    self.densify_and_split(grads, max_grad, extent)

    prune_mask = (self.get_opacity < min_opacity).squeeze()
    if max_screen_size:
        big_points_vs = self.max_radii2D > max_screen_size
        big_points_ws = self.get_scaling.max(dim=1).values > 0.1 * extent
        prune_mask = torch.logical_or(torch.logical_or(prune_mask, big_points_vs), big_points_ws)
    self.prune_points(prune_mask)
    tmp_radii = self.tmp_radii
    self.tmp_radii = None

    torch.cuda.empty_cache()



**`prune_points` 구현**:

In [ ]:
def prune_points(self, mask):
    valid_points_mask = ~mask
    optimizable_tensors = self._prune_optimizer(valid_points_mask)

    self._xyz = optimizable_tensors["xyz"]
    self._features_dc = optimizable_tensors["f_dc"]
    self._features_rest = optimizable_tensors["f_rest"]
    self._opacity = optimizable_tensors["opacity"]
    self._scaling = optimizable_tensors["scaling"]
    self._rotation = optimizable_tensors["rotation"]

    self.xyz_gradient_accum = self.xyz_gradient_accum[valid_points_mask]
    self.denom = self.denom[valid_points_mask]
    self.max_radii2D = self.max_radii2D[valid_points_mask]
    self.tmp_radii = self.tmp_radii[valid_points_mask]



**핵심**: Optimizer state도 함께 pruning

In [ ]:
def _prune_optimizer(self, mask):
    optimizable_tensors = {}
    for group in self.optimizer.param_groups:
        stored_state = self.optimizer.state.get(group['params'][0], None)
        if stored_state is not None:
            stored_state["exp_avg"] = stored_state["exp_avg"][mask]
            stored_state["exp_avg_sq"] = stored_state["exp_avg_sq"][mask]

            del self.optimizer.state[group['params'][0]]
            group["params"][0] = nn.Parameter((group["params"][0][mask].requires_grad_(True)))
            self.optimizer.state[group['params'][0]] = stored_state

            optimizable_tensors[group["name"]] = group["params"][0]
        else:
            group["params"][0] = nn.Parameter(group["params"][0][mask].requires_grad_(True))
            optimizable_tensors[group["name"]] = group["params"][0]
    return optimizable_tensors



**Adam optimizer state**:
- `exp_avg`: 1차 모멘트 (momentum)
- `exp_avg_sq`: 2차 모멘트 (RMSprop)
- Pruning 시 이들도 함께 삭제해야 메모리 누수 방지

---

### 6.5 Densification Postfix (가우시안 추가)

Clone/Split 후 새 가우시안을 기존 텐서에 추가:

In [ ]:
def densification_postfix(self, new_xyz, new_features_dc, new_features_rest, new_opacities, new_scaling, new_rotation, new_tmp_radii):
    d = {"xyz": new_xyz,
         "f_dc": new_features_dc,
         "f_rest": new_features_rest,
         "opacity": new_opacities,
         "scaling" : new_scaling,
         "rotation" : new_rotation}

    optimizable_tensors = self.cat_tensors_to_optimizer(d)
    self._xyz = optimizable_tensors["xyz"]
    self._features_dc = optimizable_tensors["f_dc"]
    self._features_rest = optimizable_tensors["f_rest"]
    self._opacity = optimizable_tensors["opacity"]
    self._scaling = optimizable_tensors["scaling"]
    self._rotation = optimizable_tensors["rotation"]

    self.tmp_radii = torch.cat((self.tmp_radii, new_tmp_radii))
    self.xyz_gradient_accum = torch.zeros((self.get_xyz.shape[0], 1), device="cuda")
    self.denom = torch.zeros((self.get_xyz.shape[0], 1), device="cuda")
    self.max_radii2D = torch.zeros((self.get_xyz.shape[0]), device="cuda")



**`cat_tensors_to_optimizer` 구현**:

In [ ]:
def cat_tensors_to_optimizer(self, tensors_dict):
    optimizable_tensors = {}
    for group in self.optimizer.param_groups:
        assert len(group["params"]) == 1
        extension_tensor = tensors_dict[group["name"]]
        stored_state = self.optimizer.state.get(group['params'][0], None)
        if stored_state is not None:
            stored_state["exp_avg"] = torch.cat((stored_state["exp_avg"], torch.zeros_like(extension_tensor)), dim=0)
            stored_state["exp_avg_sq"] = torch.cat((stored_state["exp_avg_sq"], torch.zeros_like(extension_tensor)), dim=0)

            del self.optimizer.state[group['params'][0]]
            group["params"][0] = nn.Parameter(torch.cat((group["params"][0], extension_tensor), dim=0).requires_grad_(True))
            self.optimizer.state[group['params'][0]] = stored_state

            optimizable_tensors[group["name"]] = group["params"][0]
        else:
            group["params"][0] = nn.Parameter(torch.cat((group["params"][0], extension_tensor), dim=0).requires_grad_(True))
            optimizable_tensors[group["name"]] = group["params"][0]

    return optimizable_tensors



**핵심**:
- 새 가우시안의 optimizer state는 **0으로 초기화**
- Momentum이 없으므로 처음부터 학습 시작
- 기존 가우시안의 optimizer state는 유지

---

## 7. PLY 파일 저장/로드 (PLY File Save/Load)

### 7.1 저장 형식 (Save Format)

In [ ]:
def save_ply(self, path):
    mkdir_p(os.path.dirname(path))

    xyz = self._xyz.detach().cpu().numpy()
    normals = np.zeros_like(xyz)
    f_dc = self._features_dc.detach().transpose(1, 2).flatten(start_dim=1).contiguous().cpu().numpy()
    f_rest = self._features_rest.detach().transpose(1, 2).flatten(start_dim=1).contiguous().cpu().numpy()
    opacities = self._opacity.detach().cpu().numpy()
    scale = self._scaling.detach().cpu().numpy()
    rotation = self._rotation.detach().cpu().numpy()

    dtype_full = [(attribute, 'f4') for attribute in self.construct_list_of_attributes()]

    elements = np.empty(xyz.shape[0], dtype=dtype_full)
    attributes = np.concatenate((xyz, normals, f_dc, f_rest, opacities, scale, rotation), axis=1)
    elements[:] = list(map(tuple, attributes))
    el = PlyElement.describe(elements, 'vertex')
    PlyData([el]).write(path)



**PLY 파일 구조**:

```
ply
format binary_little_endian 1.0
element vertex 100000
property float x
property float y
property float z
property float nx  (unused, always 0)
property float ny
property float nz
property float f_dc_0  (SH DC coefficient, R)
property float f_dc_1  (G)
property float f_dc_2  (B)
property float f_rest_0 (SH degree 1+, 15 × 3 = 45 properties)
...
property float f_rest_44
property float opacity (logit space)
property float scale_0 (log space)
property float scale_1
property float scale_2
property float rot_0   (quaternion, unnormalized)
property float rot_1
property float rot_2
property float rot_3
end_header
[binary data]
```



**총 속성 개수**:
- 위치: 3
- 법선 (미사용): 3
- SH features: 3 + 45 = 48
- Opacity: 1
- Scale: 3
- Rotation: 4
- **총 62 attributes**

---

### 7.2 로드 (Load from PLY)

In [ ]:
def load_ply(self, path, use_train_test_exp = False):
    plydata = PlyData.read(path)
    
    xyz = np.stack((np.asarray(plydata.elements[0]["x"]),
                    np.asarray(plydata.elements[0]["y"]),
                    np.asarray(plydata.elements[0]["z"])),  axis=1)
    opacities = np.asarray(plydata.elements[0]["opacity"])[..., np.newaxis]

    features_dc = np.zeros((xyz.shape[0], 3, 1))
    features_dc[:, 0, 0] = np.asarray(plydata.elements[0]["f_dc_0"])
    features_dc[:, 1, 0] = np.asarray(plydata.elements[0]["f_dc_1"])
    features_dc[:, 2, 0] = np.asarray(plydata.elements[0]["f_dc_2"])

    extra_f_names = [p.name for p in plydata.elements[0].properties if p.name.startswith("f_rest_")]
    extra_f_names = sorted(extra_f_names, key = lambda x: int(x.split('_')[-1]))
    assert len(extra_f_names)==3*(self.max_sh_degree + 1) ** 2 - 3
    features_extra = np.zeros((xyz.shape[0], len(extra_f_names)))
    for idx, attr_name in enumerate(extra_f_names):
        features_extra[:, idx] = np.asarray(plydata.elements[0][attr_name])
    features_extra = features_extra.reshape((features_extra.shape[0], 3, (self.max_sh_degree + 1) ** 2 - 1))



**체크포인트 복원**:
- `output/<model_id>/point_cloud/iteration_7000/point_cloud.ply` → 7000 iter 상태
- `output/<model_id>/point_cloud/iteration_30000/point_cloud.ply` → 최종 모델

---

## 8. Opacity Reset (불투명도 리셋)

opacity reset 구현:

In [ ]:
def reset_opacity(self):
    opacities_new = self.inverse_opacity_activation(torch.min(self.get_opacity, torch.ones_like(self.get_opacity)*0.01))
    optimizable_tensors = self.replace_tensor_to_optimizer(opacities_new, "opacity")
    self._opacity = optimizable_tensors["opacity"]



**동작**:
1. $\alpha_{\text{new}} = \min(\alpha_{\text{old}}, 0.01)$ → 최대 1% 불투명도
2. `inverse_sigmoid(α_new)` → logit space로 변환
3. Optimizer state 초기화

**`replace_tensor_to_optimizer` 구현**:

In [ ]:
def replace_tensor_to_optimizer(self, tensor, name):
    optimizable_tensors = {}
    for group in self.optimizer.param_groups:
        if group["name"] == name:
            stored_state = self.optimizer.state.get(group['params'][0], None)
            stored_state["exp_avg"] = torch.zeros_like(tensor)
            stored_state["exp_avg_sq"] = torch.zeros_like(tensor)

            del self.optimizer.state[group['params'][0]]
            group["params"][0] = nn.Parameter(tensor.requires_grad_(True))
            self.optimizer.state[group['params'][0]] = stored_state

            optimizable_tensors[group["name"]] = group["params"][0]
    return optimizable_tensors



**타이밍**: 3000 iter (reset_opacity_interval) 마다 수행

---

## 9. 메모리 관리 (Memory Management)

### 9.1 가우시안 수 변화 추적

Training 중 가우시안 수는 동적으로 변합니다:

In [ ]:
print(f"Number of points at initialisation : {fused_point_cloud.shape[0]}")
# 예시:
# - COLMAP sparse: ~10,000 points
# - After densification: ~100,000 - 500,000 points
# - After pruning: ~50,000 - 200,000 points (scene마다 다름)



Training 중:
- 100 iter마다: Gradient 누적
- 500 iter마다 (100-15000): Clone + Split
- 3000 iter마다: Prune + Opacity Reset

---

### 9.2 GPU 메모리 정리

Densification 후:

In [ ]:
torch.cuda.empty_cache()



**이유**:
- Clone/Split/Prune은 많은 임시 텐서 생성
- PyTorch는 자동으로 메모리 해제하지만 GPU 캐시 남음
- 명시적으로 empty_cache() 호출하여 메모리 반환

---

## 10. 실전 예시 (Practical Example)

### 10.1 초기화부터 학습까지 전체 흐름

In [ ]:
# 1. COLMAP 데이터 로드
scene_info = sceneLoadTypeCallbacks["Colmap"](args.source_path, args.images, args.eval)

# 2. GaussianModel 초기화
gaussians = GaussianModel(dataset.sh_degree)
gaussians.create_from_pcd(scene_info.point_cloud, scene_info.train_cameras, cameras_extent)

# 3. Training Setup
gaussians.training_setup(opt)

# 4. Training Loop
for iteration in range(1, opt.iterations + 1):
    # 4.1 Forward Pass
    viewpoint_cam = scene.getTrainCameras()[viewpoint_stack.pop(randint(0, len(viewpoint_stack)-1))]
    
    render_pkg = render(viewpoint_cam, gaussians, pipe, background)
    image, viewspace_point_tensor, visibility_filter, radii = render_pkg["render"], render_pkg["viewspace_points"], render_pkg["visibility_filter"], render_pkg["radii"]
    
    # 4.2 Loss Computation
    loss = (1.0 - opt.lambda_dssim) * l1_loss + opt.lambda_dssim * (1.0 - ssim_loss)
    
    # 4.3 Backward Pass
    loss.backward()
    
    # 4.4 Gradient Accumulation (for densification)
    with torch.no_grad():
        gaussians.add_densification_stats(viewspace_point_tensor, visibility_filter)
    
    # 4.5 Optimizer Step
    gaussians.optimizer.step()
    gaussians.optimizer.zero_grad(set_to_none=True)
    
    # 4.6 Learning Rate Update
    gaussians.update_learning_rate(iteration)
    
    # 4.7 Densification (500 iter마다)
    if iteration % 500 == 0 and iteration < opt.densify_until_iter:
        gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, cameras_extent, 20, radii)
    
    # 4.8 Opacity Reset (3000 iter마다)
    if iteration % 3000 == 0 and iteration < opt.densify_until_iter:
        gaussians.reset_opacity()
    
    # 4.9 SH Degree Increase (1000 iter마다)
    if iteration % 1000 == 0:
        gaussians.oneupSHdegree()
    
    # 4.10 Save Checkpoint
    if iteration in saving_iterations:
        gaussians.save_ply(os.path.join(scene.model_path, "point_cloud", f"iteration_{iteration}", "point_cloud.ply"))



---

### 10.2 가우시안 파라미터 확인

In [ ]:
# 가우시안 개수
print(f"Number of Gaussians: {gaussians.get_xyz.shape[0]}")

# 평균 스케일
mean_scale = gaussians.get_scaling.mean(dim=0)
print(f"Mean scale (x,y,z): {mean_scale}")

# 불투명도 분포
opacity = gaussians.get_opacity
print(f"Opacity range: [{opacity.min():.3f}, {opacity.max():.3f}]")
print(f"Mean opacity: {opacity.mean():.3f}")

# SH degree
print(f"Active SH degree: {gaussians.active_sh_degree}")

# 메모리 사용량
xyz_mem = gaussians._xyz.element_size() * gaussians._xyz.nelement() / (1024**2)
feat_mem = (gaussians._features_dc.element_size() * gaussians._features_dc.nelement() + 
            gaussians._features_rest.element_size() * gaussians._features_rest.nelement()) / (1024**2)
print(f"Memory: xyz={xyz_mem:.1f}MB, features={feat_mem:.1f}MB")



**출력 예시**:

```
Number of Gaussians: 156,234
Mean scale (x,y,z): tensor([0.0021, 0.0019, 0.0018], device='cuda:0')
Opacity range: [0.001, 0.995]
Mean opacity: 0.347
Active SH degree: 3
Memory: xyz=1.8MB, features=9.0MB
```



---

## 11. 디버깅 및 시각화 (Debugging and Visualization)

### 11.1 가우시안 분포 확인

In [ ]:
# 스케일 히스토그램
import matplotlib.pyplot as plt
scales = gaussians.get_scaling.detach().cpu().numpy()
plt.hist(scales.flatten(), bins=50)
plt.xlabel("Scale")
plt.ylabel("Count")
plt.title("Gaussian Scale Distribution")
plt.show()

# 큰 가우시안 비율
large_gaussians = (gaussians.get_scaling.max(dim=1).values > 0.1 * scene_extent).sum()
print(f"Large Gaussians: {large_gaussians} / {gaussians.get_xyz.shape[0]} ({100*large_gaussians/gaussians.get_xyz.shape[0]:.1f}%)")



---

### 11.2 Densification 통계

In [ ]:
# train.py에서 densification 전후
n_before = gaussians.get_xyz.shape[0]
gaussians.densify_and_prune(opt.densify_grad_threshold, 0.005, cameras_extent, 20, radii)
n_after = gaussians.get_xyz.shape[0]
print(f"Densification: {n_before} -> {n_after} ({n_after - n_before:+d})")



---

### 11.3 Gradient 분석

In [ ]:
# 높은 gradient를 가진 가우시안
grads = gaussians.xyz_gradient_accum / gaussians.denom
high_grad_mask = grads > opt.densify_grad_threshold
print(f"High gradient Gaussians: {high_grad_mask.sum()} / {grads.shape[0]}")

# 스케일별 gradient 분포
small_scale_mask = gaussians.get_scaling.max(dim=1).values <= opt.percent_dense * scene_extent
large_scale_mask = ~small_scale_mask
print(f"Clone candidates: {torch.logical_and(high_grad_mask.squeeze(), small_scale_mask).sum()}")
print(f"Split candidates: {torch.logical_and(high_grad_mask.squeeze(), large_scale_mask).sum()}")



---